# Models Point-E And Shape-E
## Google Colab — Runtime Actual + A100


---
## Célula 1 — Instalar Point-E

In [1]:
!pip install git+https://github.com/openai/shap-e.git

  Cloning https://github.com/openai/shap-e.git to /tmp/pip-req-build-53pcdoyf
  Running command git clone --filter=blob:none --quiet https://github.com/openai/shap-e.git /tmp/pip-req-build-53pcdoyf
  Resolved https://github.com/openai/shap-e.git to commit 50131012ee11c9d2617f3886c10f000d3c7a3b43
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-install-g2ogx09h/clip_50a59d0c2ea34fcba245bdda87510a10
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-install-g2ogx09h/clip_50a59d0c2ea34fcba245bdda87510a10
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.8 MB/s eta 0:00:00
  Created wheel for shap-e: filename=shap_e-0.0.0-py3-none-any.whl size=133335 sha256=3a1074f36edb35

In [2]:
import os

# Limpar montagem antiga que ficou presa
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    # Tenta desmontar primeiro
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except:
        pass
    # Se ainda tem arquivos, remove o diretório
    import shutil
    shutil.rmtree('/content/drive', ignore_errors=True)

from google.colab import drive
drive.mount('/content/drive')

# Configurar cache no Drive
PROJECT_DIR = '/content/drive/MyDrive/Ufma/ECP/CG'
# os.environ['XDG_CACHE_HOME'] = PROJECT_DIR
# os.makedirs(PROJECT_DIR + '/cache_modelos', exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"✓ Diretório: {os.getcwd()}")

Mounted at /content/drive
✓ Diretório: /content/drive/MyDrive/Ufma/ECP/CG


---
## Configurar modelo

In [3]:
# Inferência Point-E
import torch
from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config
from shap_e.util.notebooks import decode_latent_mesh

In [4]:
device = torch.device('cuda')
xm = load_model('transmitter', device=device)
model = load_model('image300M', device=device)
diffusion = diffusion_from_config(load_config('diffusion'))

/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:31: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:43: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:61: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:86: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


  0%|          | 0.00/1.78G [00:00<?, ?iB/s]

100%|███████████████████████████████████████| 890M/890M [00:12<00:00, 77.2MiB/s]


  0%|          | 0.00/1.26G [00:00<?, ?iB/s]

In [5]:
print(f"✓ Diretório: {os.getcwd()}")

pasta = '/content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs'
print(os.listdir(pasta))

✓ Diretório: /content/drive/MyDrive/Ufma/ECP/CG
['leao.jpg', 'busto.jpg', 'poste.jpeg', 'canhao.jpeg', 'edited']


In [6]:

def criar_modelos_ply(latents,name):
  # 6. (Opcional) Converter point cloud para mesh com modelo SDF
   # Decodificar para mesh
    for latent in latents:
        t = decode_latent_mesh(xm, latent).tri_mesh()
        with open(name, 'wb') as f:
            t.write_ply(f)
            print("Mesh salvo em "+name)
    # # Salvar mesh
    # with open(name, 'wb') as f:
    #     mesh.write_ply(f)
    # print("Mesh salvo em "+name)

In [11]:
name  = os.path.splitext(os.path.basename("/content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/busto-edited.png"))[0]
print(PROJECT_DIR)
print(name)

/content/drive/MyDrive/Ufma/ECP/CG
busto-edited


In [13]:
import glob, subprocess, sys
from shap_e.util.image_util import load_image



imgs = sorted(glob.glob(pasta+"/edited/*.*"))

print(f"Imagens encontradas: {len(imgs)}")
for img in imgs:
    print(f"  - {img}")

for img in imgs:
    print(f"\n{'='*50}")
    print(f"Processando: {img}")
    print(f"{'='*50}", flush=True)
    path=img
    # img = Image.open(path)  # Sua imagem de entrada
    image = load_image(path)

    latents = sample_latents(
        batch_size=1,
        model=model,
        diffusion=diffusion,
        model_kwargs=dict(images=[image]),
        guidance_scale=3.0,
        progress=True,
        clip_denoised=True,
        use_fp16=True,
        use_karras=True,
        karras_steps=64,
        sigma_min=1e-3,
        sigma_max=160,
        s_churn=0,
    )

    name = name  = os.path.splitext(os.path.basename(path))[0] + '_shap_e.ply'
    criar_modelos_ply(latents,name)



print(f"\n✓ Processamento concluído!")

Imagens encontradas: 4
  - /content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/busto-edited.png
  - /content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/canhao-edited.png
  - /content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/leao-edited.png
  - /content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/poste-edited.png

Processando: /content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/busto-edited.png


  0%|          | 0/64 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/shap_e/models/stf/renderer.py:286: UserWarning:

exception rendering with PyTorch3D: No module named 'pytorch3d'

/usr/local/lib/python3.12/dist-packages/shap_e/models/stf/renderer.py:287: UserWarning:

falling back on native PyTorch renderer, which does not support full gradients



Mesh salvo em busto-edited_shap_e.ply

Processando: /content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/canhao-edited.png


  0%|          | 0/64 [00:00<?, ?it/s]

Mesh salvo em canhao-edited_shap_e.ply

Processando: /content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/leao-edited.png


  0%|          | 0/64 [00:00<?, ?it/s]

Mesh salvo em leao-edited_shap_e.ply

Processando: /content/drive/MyDrive/Ufma/ECP/CG/One-2-3-45-master/inputs/edited/poste-edited.png


  0%|          | 0/64 [00:00<?, ?it/s]

Mesh salvo em poste-edited_shap_e.ply

✓ Processamento concluído!


In [9]:
!pip install trimesh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 28.1 MB/s eta 0:00:00


In [15]:

import trimesh
import plotly.graph_objects as go
import numpy as np
imgs = sorted(glob.glob(PROJECT_DIR+"/modelos_shap_e/*.*"))

print(f"Imagens encontradas: {len(imgs)}")
for img in imgs:
    print(f"  - {img}")
    mesh_path=img
    mesh = trimesh.load_mesh(mesh_path)
    vertices = mesh.vertices
    faces = mesh.faces

    # Extrair cores dos vértices (RGBA -> RGB normalizado)
    if mesh.visual.vertex_colors is not None:
        colors = mesh.visual.vertex_colors[:, :3]  # RGB
        # Converter para string de cor por vértice
        vertex_colors = ['rgb({},{},{})'.format(r, g, b) for r, g, b in colors]

        fig = go.Figure(data=[go.Mesh3d(
            x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            vertexcolor=colors,
            opacity=1.0
        )])
    else:
        fig = go.Figure(data=[go.Mesh3d(
            x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            color='lightblue', opacity=0.9
        )])

    fig.update_layout(scene=dict(aspectmode='data'))
    fig.show()

Output hidden; open in https://colab.research.google.com to view.